# 🦜🔗 LangChain Revision Notes: Messages & Conversation Memory

## 1. What are Messages? (Beginner-Friendly Definition)

Think of a conversation with an AI like a **WhatsApp or Slack chat thread**. Every single turn in the conversation has:
1. **Role (Who said it?)**: System (instructions), Human (user), AI (assistant), or Tool (function result).
2. **Content (What was said?)**: The actual text, code, image, or output.
3. **Metadata (Extra Details)**: Optional information like user name, message ID, or token usage.

In LangChain, **Messages** are standard Python objects that package this data together so that **any AI model** (OpenAI, Gemini, Groq) understands the conversation history consistently.

---

### 🛠️ Step 1: Model Setup
**What this code does**: Loads your secret API key from environment variables and initializes the chat model (`openai:gpt-4.1` or `groq:qwen/qwen3.8-27b`) using LangChain's unified `init_chat_model()` function.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
# model = init_chat_model("groq:qwen/qwen3.8-27b")
model = init_chat_model("openai:gpt-4.1")

---
## 2. Text Prompts vs. Message Prompts

### 🅰️ Text Prompts (Simple String Queries)

- **What is a Text Prompt?**: Passing a plain Python string directly to `model.invoke("Hello")`.
- **Analogy**: Like asking a quick one-off question to a stranger on the street ("What time is it?"). They answer, and the interaction ends.

#### 💡 When to use Text Prompts:
- ✅ Single, standalone questions.
- ✅ Tasks where past conversation memory is NOT needed.
- ✅ Quick testing with minimal code.

#### 🔹 Example 1: Asking a Standalone Question
**Code Explanation**: We pass a simple string directly into `model.invoke()`. The AI processes the query and returns a complete text response.

In [3]:
model.invoke("Please tell me what is Artificial Intelligence?")

#### 🔹 Example 2: Another Simple Text Prompt
**Code Explanation**: Demonstrating another quick one-off query without any setup or conversation context.

In [4]:
model.invoke("What is langchain?")

---
### 🅱️ Message Prompts (Structured List of Message Objects)

- **What is a Message Prompt?**: Passing a Python list of message objects (`model.invoke([SystemMessage(...), HumanMessage(...)])`).
- **Analogy**: Like a structured conversation where you set ground rules first ("Act as my math tutor"), ask questions, and build on previous replies.

--- 
## 3. Different Types of Messages (Beginner Definitions)

LangChain provides 4 main message classes:

### 1️⃣ `SystemMessage` (The Rulebook / Persona)
- **Definition**: Instructions given to the AI *before* the conversation starts to define its behavior, tone, persona, and output rules.
- **Real-World Analogy**: Telling an actor, *"You play the role of a helpful Python Senior Developer."*
- **Example**:
  ```python
  from langchain.messages import SystemMessage
  sys_msg = SystemMessage("You are a helpful coding assistant that speaks concisely.")
  ```

### 2️⃣ `HumanMessage` (The User's Question / Prompt)
- **Definition**: Represents anything sent by the human user (questions, text, images, uploaded files).
- **Real-World Analogy**: What you type into the chat box.
- **Example**:
  ```python
  from langchain.messages import HumanMessage
  user_msg = HumanMessage(content="How do I create a REST API?", name="alice")
  ```

### 3️⃣ `AIMessage` (The AI's Answer / Response)
- **Definition**: The response generated by the AI model. It contains the text answer, tool call requests, and token metadata.
- **Real-World Analogy**: The AI's reply popping up on your screen.
- **Example**:
  ```python
  from langchain.messages import AIMessage
  ai_msg = AIMessage("I would be happy to help you with that!")
  ```

### 4️⃣ `ToolMessage` (The Tool's Execution Result)
- **Definition**: The output returned after executing a tool/function (e.g. weather search, database query) sent back to the AI.
- **Key Rule**: Must include a `tool_call_id` matching the ID generated by `AIMessage` so the AI links the result to its question.
- **Example**:
  ```python
  from langchain.messages import ToolMessage
  tool_msg = ToolMessage(content="Sunny, 72°F", tool_call_id="call_123")
  ```

--- 
### 📊 Quick Comparison Matrix

| Message Type | Who Sends It? | What is its Purpose? | Easy Analogy |
| :--- | :--- | :--- | :--- |
| **`SystemMessage`** | System / You | Sets rules, persona, & guidelines | Job description given to AI |
| **`HumanMessage`** | Human User | User prompt or question | Your message in a chat |
| **`AIMessage`** | AI Model | AI response or tool call request | AI's reply in a chat |
| **`ToolMessage`** | Python Function | Sends execution results back to AI | Answer returned from a tool |

---

### 💡 Code Examples & Detailed Walkthroughs

#### 🔹 Example 1: `SystemMessage` + `HumanMessage` (Setting a Persona)
**Code Explanation**: We pass a list containing `SystemMessage("You are poetry expert")` and `HumanMessage(...)`. The system message forces the AI to adopt the persona of a poet when answering.

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are poetry expert"),
    HumanMessage("Write a poem on Artificial Intelligence")
]
response = model.invoke(messages)
response.content

#### 🔹 Example 2: Simple Coding Assistant Persona
**Code Explanation**: Creating a `sys_msg` variable set to *"You are a helpful coding assistant"* and combining it with a `HumanMessage` query.

In [7]:
sys_msg = SystemMessage("You are a helpful coding assistant.")

messages=[
    sys_msg,
    HumanMessage("How do i create a REST API in python?")
]

response = model.invoke(messages)
print(response.content)

#### 🔹 Example 3: Detailed Multi-line System Instructions
**Code Explanation**: A detailed system prompt instructing the AI to act as a *"Senior Python Developer"*, always provide code examples, and explain reasoning clearly.

In [10]:
# Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

sys_msg = SystemMessage("""
You are a Senior Python Developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but through in your explanations.
""")

messages = [
    sys_msg,
    HumanMessage("How do i create a REST API?")
]

response = model.invoke(messages)
print(response.content)

---
### 🏷️ Message Metadata (`name` & `id`)

**Code Explanation**: `HumanMessage` accepts optional metadata fields:
- `name="alice"`: Identifies which user sent the message in multi-person group chats.
- `id="msg_123"`: Assigns a unique tracking ID for debugging and tracing logs (e.g. in LangSmith).

In [ ]:
## Messages metadata
human_msg = HumanMessage(
    content="Hello",
    name="alice", # Optional: Identifies different users
    id="msg_123" # Optional: Uniques identifiers for tracing
)

#### 🔹 Invoking the Model with Metadata
**Code Explanation**: Passing `[human_msg]` to `model.invoke()`. The model receives both the content (`"Hello"`) and metadata attributes.

In [12]:
response = model.invoke([human_msg])
response

---
### 💬 Building Conversation Memory (Manual `AIMessage` Injection)

**Code Explanation**: LLMs are stateless (they forget everything after each call). To create a chat memory, we manually create an `AIMessage` representing a past AI response and place it inside the `messages` list (`[SystemMessage, HumanMessage, ai_msg, HumanMessage]`). The AI reads the full history and understands the context for the next turn (*"What's 2+2?"*).

In [3]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

# Create an AI Message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to coversation history
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Can you help me?"),
    ai_msg, #Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

#### 🔹 Token Usage Tracking (`response.usage_metadata`)
**Code Explanation**: `response.usage_metadata` returns a dictionary showing `input_tokens` (prompt cost), `output_tokens` (response cost), and `total_tokens` consumed by the LLM call.

In [4]:
response.usage_metadata

---
### 🔧 Tool Execution & ToolMessage Trajectory (Important: Tool ID Should Be Same!)

**Code Explanation**: Demonstrating how tool calls flow through messages:
1. `ai_msg` represents the AI requesting a tool call (`get_weather` for San Francisco with `id="call_123"`).
2. `tool_msg = ToolMessage(content="Sunny, 72°F", tool_call_id="call_123")` contains the execution output.
3. ⚠️ **CRITICAL RULE (Tool ID Should Be Same)**: The `tool_call_id` in `ToolMessage` **MUST BE THE EXACT SAME** as the `id` generated in `ai_msg.tool_calls` (`"call_123"`). This is how the AI model pairs the result with its original question.

In [ ]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the message for brevity)
ai_msg = AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location": "San Francisco"},
        "id":"call_123"
    }]
)

# Execute tool and create result message
weather_result ="Sunny, 72°F"
tool_msg= ToolMessage(
    content=weather_result,
    tool_call_id="call_123" # Must match the call ID
)

# Continue conversation
messages=[
    HumanMessage("What's the weather in San Franscisco?"),
    ai_msg, # Model's tool call
    tool_msg, # Tool execution result
]

response = model.invoke(messages) # Model process the result
print(tool_msg)
print(response)